# 01 — The `Leaf`, in pictures

Hands-on companion to [`docs/01-leaf.md`](../docs/01-leaf.md).

> Run this with the notebook extra installed: `pip install -e ".[notebooks]"`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from bonsaigrad import Leaf
from utils import backward_levels, draw_graph

## An expression grows a graph

Writing `f = a * b + a` does not just compute a number — every `*` and `+` creates a new `Leaf` that remembers its stems. Build it one operation at a time and watch the tree grow:

```
g = a * b        # step 1
f = g + a        # step 2
```

Notice `a` is used **twice** — once inside the product, once on its own. It shows up as a single node with two arrows leaving it.

In [ ]:
a = Leaf(3.0)
b = Leaf(4.0)

g = a * b
f = g + a

names = {id(a): "a", id(b): "b", id(g): "g", id(f): "f"}

steps = [(g, "g = a*b"), (f, "f = g + a")]
fig, axes = plt.subplots(1, len(steps), figsize=(9, 4.0))
for ax, (apex, label) in zip(axes, steps):
    draw_graph(apex, names, ax=ax, title=label, flow="up")
plt.tight_layout()
plt.show()

## Backward pass

`wire` seeds the apex with `1` and walks the tree in reverse. Here we freeze it after each level to watch the gradient cascade — a box is highlighted once its gradient is known, and an arrow turns red once gradient has actually crossed it.

Keep your eye on `a`. It receives gradient **twice**: first straight from `f` (step 1, `a.grad = 1`), then again through the product `g` (step 2, `+= b = 4`). The two paths add up to `a.grad = 5` — that is the `+=` in the backward rules, made visible.

In [ ]:
levels = backward_levels(f)

f.rest()  # clean slate
f.grad = np.ones_like(f.data)  # seed the apex
fired = set()  # nodes whose _backward has run

fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.0))
draw_graph(f, names, ax=axes[0], title="seed: f.grad = 1", fired=set())
for step, level in enumerate(levels[:-1], start=1):  # roots have no stems
    for node in level:
        node._backward()
        fired.add(id(node))
    depth = len(levels) - step
    draw_graph(f, names, ax=axes[step],
               title=f"step {step}: pushed down from depth {depth}",
               fired=fired)
plt.tight_layout()
plt.show()

print("a.grad =", a.grad.item(), "(expected 5.0)")
print("b.grad =", b.grad.item(), "(expected 3.0)")
assert a.grad.item() == 5.0 and b.grad.item() == 3.0

## A tiny neuron

The same machinery already gives us a single neuron: `y = w * x + b`. Weight `w`,
input `x`, bias `b`. Below, values travelling up, then gradient travelling down.

By hand, `∂y/∂w = x`, `∂y/∂x = w`, `∂y/∂b = 1` — read them straight off the
highlighted boxes on the right.

In [ ]:
w, x, b = Leaf(2.0), Leaf(-3.0), Leaf(1.0)
y = w * x + b
names = {id(w): "w", id(x): "x", id(b): "b", id(y): "y"}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.2))
draw_graph(y, names, ax=ax1, title="Forward — values flow up", flow="up")
y.wire()
draw_graph(y, names, ax=ax2, title="Backward — gradient flows down")
plt.tight_layout()
plt.show()

print(f"y = {y.data.item()}  |  w.grad = {w.grad.item()}  x.grad = {x.grad.item()}  b.grad = {b.grad.item()}")

## Full example

`L = (a + b) * (b + c)` — two sums feeding one product. `b` is used **twice**, so it shows up as a single node with two arrows leaving it, one into each sum.

The full round trip, side by side. **Left:** the forward pass, values climbing from the roots to `L = 35`. **Right:** `wire` seeding `L.grad = 1` and pushing gradient back down. Each `*` edge carries its **local derivative** `∂node/∂stem` — the other operand — which the gradient is multiplied by on the way down; `+` edges pass it through unchanged, so they carry no label.

Because `b` feeds both sums, it collects gradient from both paths: `7 + 5 = 12`.

In [ ]:
a, b, c = Leaf(2.0), Leaf(3.0), Leaf(4.0)
L = (a + b) * (b + c)

names = {id(a): "a", id(b): "b", id(c): "c", id(L): "L"}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.2))
draw_graph(L, names, ax=ax1, title="Forward — values flow up", flow="up")
L.wire()
draw_graph(L, names, ax=ax2, title="Backward — gradient flows down", edge_labels=True)
plt.tight_layout()
plt.show()

print("L =", L.data.item())
for name, leaf in zip("abc", (a, b, c)):
    print(f"{name}.grad = {leaf.grad.item():.0f}")